In [7]:
import pandas as pd

# 1. Membaca dataset dari file CSV
df = pd.read_csv("ssh_anomaly_dataset.csv")

# 2. Membersihkan nama kolom (menghilangkan spasi berlebih)
df.columns = df.columns.str.strip()

# 3. Menampilkan ukuran data (jumlah baris dan kolom)
print("Ukuran dataset:", df.shape)

# 4. Menampilkan 5 baris pertama dalam bentuk tabel yang rapi
display(df.head())

Ukuran dataset: (41825, 7)


,timestamp,source_ip,username,event_type,status,label,detail
0,2025-06-15 09:00:00.000000,192.168.0.46,john_doe,Accepted password,success,normal,NaN
1,2025-06-15 09:00:01.673608,192.168.0.46,john_doe,Command executed,pwd,normal,NaN
2,2025-06-15 09:01:05.018929,192.168.0.37,analyst_user,Accepted password,success,normal,NaN
3,2025-06-15 09:01:09.198906,192.168.0.37,analyst_user,Command executed,uptime,normal,NaN
4,2025-06-15 09:04:30.880364,192.168.0.46,john_doe,Disconnected,normal_logout,normal,NaN


In [12]:
import numpy as np
from sklearn.preprocessing import LabelEncoder

# 1. Menghapus kolom yang tidak relevan
df_bersih = df.drop(['timestamp', 'detail'], axis=1)
df_bersih.dropna(inplace=True)

# 2. BARU: Menghapus kelas 'config_anomaly' yang cuma ada 2 baris 
# (Agar algoritma SMOTE tidak error secara matematis)
df_bersih = df_bersih[df_bersih['label'] != 'config_anomaly']

# 3. Mengubah data teks menjadi angka
encoder_ip = LabelEncoder()
encoder_user = LabelEncoder()
encoder_event = LabelEncoder()
encoder_status = LabelEncoder()

df_bersih['source_ip'] = encoder_ip.fit_transform(df_bersih['source_ip'])
df_bersih['username'] = encoder_user.fit_transform(df_bersih['username'])
df_bersih['event_type'] = encoder_event.fit_transform(df_bersih['event_type'])
df_bersih['status'] = encoder_status.fit_transform(df_bersih['status'])

encoders = {
    'ip': encoder_ip, 
    'user': encoder_user, 
    'event': encoder_event, 
    'status': encoder_status
}

print("Ukuran dataset setelah dibersihkan:", df_bersih.shape)
print("\nJumlah masing-masing serangan/label:")
print(df_bersih['label'].value_counts())

Ukuran dataset setelah dibersihkan: (41823, 5)

Jumlah masing-masing serangan/label:
label
brute_force                     37749
normal                           3707
brute_force_connection_issue      367
Name: count, dtype: int64


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

X = df_bersih.drop('label', axis=1)
y = df_bersih['label']

# Menambahkan stratify=y agar pembagian kelas yang kecil tetap rata dan adil
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Menyeimbangkan data dengan SMOTE... (Tunggu sebentar)")
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_smote)
X_test_scaled = scaler.transform(X_test)

print("Data seimbang dan siap dimasukkan ke dalam Otak AI!")

Menyeimbangkan data dengan SMOTE... (Tunggu sebentar)
Data seimbang dan siap dimasukkan ke dalam Otak AI!


In [15]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Membangun "Otak" Jaringan Syaraf Tiruan
# Kita pakai 2 lapis neuron (64 dan 32) sesuai arsitektur di laporanmu
ann_model = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=50, random_state=42, verbose=True)

print("AI sedang belajar mengenali pola serangan SSH... (Proses ini memakan waktu 1-3 menit)")

# 2. Proses Belajar (Training) menggunakan data yang sudah di-SMOTE
ann_model.fit(X_train_scaled, y_train_smote)

print("\nAI selesai belajar! Sekarang mari kita uji kemampuannya.")

# 3. Proses Ujian (Testing) dengan data uji yang belum pernah dilihat AI
y_pred = ann_model.predict(X_test_scaled)

# 4. Menampilkan Hasil Rapor AI
print("\n================ HASIL EVALUASI ================")
print("Akurasi Keseluruhan :", accuracy_score(y_test, y_pred))
print("\nLaporan Klasifikasi (Precision, Recall, F1-Score):")
# AI akan otomatis memunculkan metrik untuk 'brute_force', 'normal', dll.
print(classification_report(y_test, y_pred))

AI sedang belajar mengenali pola serangan SSH... (Proses ini memakan waktu 1-3 menit)
Iteration 1, loss = 0.12403281
Iteration 2, loss = 0.00143854
Iteration 3, loss = 0.00040105
Iteration 4, loss = 0.00020023
Iteration 5, loss = 0.00012225
Iteration 6, loss = 0.00008772
Iteration 7, loss = 0.00006712
Iteration 8, loss = 0.00005554
Iteration 9, loss = 0.00004777
Iteration 10, loss = 0.00004312
Iteration 11, loss = 0.00004049
Iteration 12, loss = 0.00003768
Iteration 13, loss = 0.00003578
Iteration 14, loss = 0.00003439
Iteration 15, loss = 0.00003343
Training loss did not improve more than tol=0.000100 for 10 consecutive epochs. Stopping.

AI selesai belajar! Sekarang mari kita uji kemampuannya.

================ HASIL EVALUASI ================
Akurasi Keseluruhan : 1.0

Laporan Klasifikasi (Precision, Recall, F1-Score):
                              precision    recall  f1-score   support

                 brute_force       1.00      1.00      1.00      7550
brute_force_connection_iss

In [16]:
import joblib

# Menyimpan Model AI
joblib.dump(ann_model, 'ai_bruteforce_model.pkl')

# Menyimpan Scaler
joblib.dump(scaler, 'scaler.pkl')

# Menyimpan SEMUA Encoder teks tadi
joblib.dump(encoders, 'encoders.pkl')

print("Hore! Otak AI SSH yang baru beserta alat terjemahannya berhasil disimpan.")

Hore! Otak AI SSH yang baru beserta alat terjemahannya berhasil disimpan.
